# Solar Prediction Colab Horizon Sweep (Hugging Face)

Run the package on the full dataset downloaded from a Hugging Face Dataset repository across several forecast horizons, and upload all checkpoints, metrics, and plots to a Hugging Face Model repository.

Set `HF_TOKEN` in Colab Secrets (`userdata`) or `os.environ['HF_TOKEN']` before running.

In [ ]:
GIT_REPO_URL = 'https://github.com/itxtx/solar_prediction.git'
PROJECT_DIR = '/content/solar_prediction'

# Hugging Face Hub repositories
HF_DATASET_REPO = 'itxtx/solar-prediction-data'
HF_DATASET_FILENAME = 'solar_weather.csv'
HF_MODEL_REPO = 'itxtx/solar-prediction-artifacts'
HF_PATH_IN_REPO = 'colab_horizon_sweep_actual_data'
UPLOAD_TO_HF = True

DATA_PATH = 'data/solar_weather.csv'
OUTPUT_ROOT = 'artifacts/colab_horizon_sweep_actual_data'

EPOCHS = 50
HIDDEN_DIM = 32
BATCH_SIZE = 256
SEASONAL_LAG = 96  # Daily cycle for 15-minute data.
HORIZONS = {
    '15min': 1,
    '1h': 4,
    '4h': 16,
    '24h': 96,
}


In [ ]:
import os
import subprocess
from pathlib import Path

if not (Path.cwd() / 'pyproject.toml').exists():
    project = Path(PROJECT_DIR)
    if not project.exists():
        subprocess.run(['git', 'clone', '--depth', '1', GIT_REPO_URL, str(project)], check=True)
    os.chdir(project)

print('Working directory:', Path.cwd())


Use a fresh Colab runtime. This notebook installs `huggingface_hub` and the local package without touching Colab's scientific Python stack.

In [ ]:
%pip install -q huggingface_hub
%pip install -q --no-deps -e .


In [ ]:
from solar_prediction.hf_hub import download_dataset_file, upload_artifacts_folder

data_path = Path(DATA_PATH)
if not data_path.exists():
    print(f'Downloading {HF_DATASET_FILENAME} from Hugging Face dataset {HF_DATASET_REPO}...')
    data_path = download_dataset_file(
        HF_DATASET_REPO,
        HF_DATASET_FILENAME,
        local_dir=data_path.parent,
    )

assert data_path.exists(), f'Full dataset not found: {data_path}'
print('Full dataset:', data_path)


In [ ]:
import pandas as pd

preview = pd.read_csv(data_path, nrows=5)
time_col = pd.read_csv(data_path, usecols=['Time'])['Time']
parsed_time = pd.to_datetime(time_col, errors='coerce')

print('Rows:', len(time_col))
print('Columns:', preview.columns.tolist())
print('Time range:', parsed_time.min(), 'to', parsed_time.max())
preview.head()


In [ ]:
from io import StringIO
import shlex
import subprocess

import pandas as pd
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
all_metrics = []

for label, horizon_steps in HORIZONS.items():
    output_dir = f'{OUTPUT_ROOT}/{label}'
    cmd = [
        'solar-predict',
        'compare',
        '--data', str(data_path),
        '--epochs', str(EPOCHS),
        '--hidden-dim', str(HIDDEN_DIM),
        '--batch-size', str(BATCH_SIZE),
        '--horizon-steps', str(horizon_steps),
        '--seasonal-lag', str(SEASONAL_LAG),
        '--device', device,
        '--output', output_dir,
        '--quiet',
    ]

    print('\nHorizon:', label, f'({horizon_steps} steps)')
    print('Command:', shlex.join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')

    output_lines = result.stdout.splitlines()
    header_idx = next(i for i, line in enumerate(output_lines) if line.startswith('model,'))
    metrics = pd.read_csv(StringIO('\n'.join(output_lines[header_idx:])))
    metrics.insert(0, 'horizon', label)
    metrics.insert(1, 'horizon_steps', horizon_steps)
    all_metrics.append(metrics)

sweep_metrics = pd.concat(all_metrics, ignore_index=True)
output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)
sweep_metrics.to_csv(output_root / 'horizon_sweep_metrics.csv', index=False)
sweep_metrics


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
horizon_order = list(HORIZONS)
rmse_table = sweep_metrics.pivot(index='horizon', columns='model', values='rmse').loc[horizon_order]
r2_table = sweep_metrics.pivot(index='horizon', columns='model', values='r2').loc[horizon_order]

rmse_table.plot.bar(ax=axes[0])
axes[0].set_title('RMSE by horizon')
axes[0].set_xlabel('')
axes[0].set_ylabel('W/m^2')

r2_table.plot.bar(ax=axes[1])
axes[1].set_title('R2 by horizon')
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].legend(loc='lower left')

plt.tight_layout()
fig.savefig(output_root / 'horizon_sweep_plot.png', dpi=150)
plt.show()

if UPLOAD_TO_HF:
    commit_url = upload_artifacts_folder(
        output_root,
        HF_MODEL_REPO,
        path_in_repo=HF_PATH_IN_REPO,
        commit_message='Upload horizon sweep artifacts',
    )
    print('Uploaded artifacts to Hugging Face:', commit_url)

best_by_horizon = sweep_metrics.sort_values('rmse').groupby('horizon', sort=False).first()
best_by_horizon[['model', 'rmse', 'mae', 'r2', 'capped_mape']]
